# Task 7 — JPEG compression robustness
Wpływ kompresji JPEG na dokładność systemu. ≥3 poziomy jakości.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, cv2, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.degradation import apply_jpeg, JPEG_QUALITIES
from src.metrics import compute_far_frr
from src.utils import list_images, psnr

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()
THRESHOLD = 0.4
TEST_DIR = Path('../data/test')

In [ ]:
all_images = []
for user_dir in TEST_DIR.iterdir():
    if user_dir.name not in db.get_all_users(): continue
    for p in list_images(user_dir):
        img = cv2.imread(str(p))
        if img is not None: all_images.append((user_dir.name, img))

results = []
for q in JPEG_QUALITIES:
    scores, labs, psnr_vals = [], [], []
    for user_id, img in all_images:
        compressed = apply_jpeg(img, q)
        psnr_vals.append(psnr(img, compressed))
        emb = get_embedding(app, compressed)
        if emb is None: continue
        refs = db.get_user_embeddings(user_id)
        score = max(cosine_similarity(emb, r) for r in refs)
        scores.append(score); labs.append(1)
    far, frr, acc = compute_far_frr(np.array(scores), np.array(labs), THRESHOLD)
    mean_psnr = np.mean(psnr_vals)
    results.append({'quality': q, 'mean_PSNR': mean_psnr, 'FAR': far, 'FRR': frr, 'Acc': acc})
    print(f'JPEG q={q:2d} | PSNR={mean_psnr:.1f} dB | FAR={far:.4f} FRR={frr:.4f} Acc={acc:.4f}')

df = pd.DataFrame(results)
df.to_csv('../results/task7/jpeg_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['quality'], df['Acc'], marker='o')
axes[0].set_xlabel('JPEG Quality'); axes[0].set_ylabel('Accuracy'); axes[0].set_title('Accuracy vs JPEG Quality')
axes[1].plot(df['quality'], df['mean_PSNR'], marker='s', color='orange')
axes[1].set_xlabel('JPEG Quality'); axes[1].set_ylabel('Mean PSNR [dB]'); axes[1].set_title('PSNR vs JPEG Quality')
plt.tight_layout()
plt.savefig('../results/task7/jpeg_plot.png', dpi=150)
plt.show()